# Lab 1 — Good tools + plan for failure (EP06)

Two design principles from §19.3 that work together:
- **③ Provide good tools** — the model sees only the name, the description and the argument types → a vaguely written tool gets misused
- **④ Plan for failure** — every tool call can fail → retry, fallback and graceful degradation live in the **harness**, not in the model's head

**What you'll learn**
1. A vague tool vs a clear tool — 5 runs each, counting how often the model uses it correctly
2. When the system fails: a bare `"Error"` vs an error that states the cause
3. Retry in the harness — the model never even knows it failed
4. Graceful degradation — answer from a cache and tell the user plainly

The paper library, the failing search API and the cache are **simulated data written for teaching**.
Every tool call and model answer comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §19.3 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **6 places marked `✏️ TODO`** (replace `___` or `...` with your code) — each one gives the time in the video where that code is explained (▶ m:ss). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep06-lab1-good-tools.en.ipynb`** (the same code as in the video).

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs well under one cent) |
| Packages | `openai`, `python-dotenv` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

## 1. Calling Gemma through OpenRouter

**Why `google/gemma-4-26b-a4b-it`** — it follows instructions in Thai well and **can call tools** (tool calling),
which every lab in EP06 needs

**Why the provider is pinned to DeepInfra** — the same one as EP03, which we tested for tool calling and repeatable results
(different providers can give different results even under the same model name — EP01 Lab 3)

**Not free**, but very cheap: DeepInfra charges USD 0.07 per million input tokens and USD 0.34 per million output tokens
— at the end of the lab we add up the tokens actually used
· The `:free` version doesn't work (tested 2026-11-12: 3 out of 3 calls got 429 "rate-limited upstream")

In [ ]:
import json, os, time
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())

In [ ]:
MODEL = "google/gemma-4-26b-a4b-it"
PROVIDER = {"order": ["deepinfra/fp8"], "allow_fallbacks": False}
EXTRA = {"reasoning": {"enabled": False}, "provider": PROVIDER}

## 2. A simulated paper library + the agent loop

The library has 12 papers (written for teaching — the titles are real papers, but the library and ids are ours)
`find_papers` is the "search system" behind the tool: it matches words in the title and filters by year

`run_agent` is the agent loop from EP03 Lab 1 (call the model → if it asks for a tool, run it and loop → until it answers)
plus a token counter and a record of what each tool call was given and what came back

In [ ]:
PAPERS = [
    {"id": "P01", "title": "Informer: efficient transformer for long sequence time-series forecasting", "year": 2021},
    {"id": "P02", "title": "Autoformer: decomposition transformers for long-term series forecasting", "year": 2021},
    {"id": "P03", "title": "FEDformer: frequency enhanced decomposed transformer for time series", "year": 2022},
    {"id": "P04", "title": "Are transformers effective for time series forecasting?", "year": 2023},
    {"id": "P05", "title": "PatchTST: a time series is worth 64 words", "year": 2023},
    {"id": "P06", "title": "iTransformer: inverted transformers for time series forecasting", "year": 2024},
    {"id": "P07", "title": "TimesFM: a decoder-only foundation model for time-series forecasting", "year": 2024},
    {"id": "P08", "title": "Chronos: learning the language of time series", "year": 2024},
    {"id": "P09", "title": "Attention is all you need", "year": 2017},
    {"id": "P10", "title": "BERT: pre-training of deep bidirectional transformers", "year": 2019},
    {"id": "P11", "title": "Temporal fusion transformers for interpretable time series forecasting", "year": 2019},
    {"id": "P12", "title": "Moirai: unified training of universal time series forecasting transformers", "year": 2024},
]
STOP = {"with", "from", "paper", "papers", "model", "models", "using", "published", "since", "research", "about"}

def find_papers(query, year_from=0, library=PAPERS):
    words = [w for w in query.lower().replace("-", " ").split() if len(w) > 3 and w not in STOP]
    return [p for p in library
            if p["year"] >= year_from and any(w in p["title"].lower().replace("-", " ") for w in words)]

In [ ]:
SYSTEM = "You are the Cogito Learn paper-search assistant. Always use tools to get real data; never answer from memory."
USAGE = {"in": 0, "out": 0}

def run_agent(question, tools, run_tool, max_steps=6, show=True):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    calls = []
    for step in range(max_steps):
        reply = client.chat.completions.create(model=MODEL, extra_body=EXTRA, messages=messages, tools=tools)
        USAGE["in"] += reply.usage.prompt_tokens
        USAGE["out"] += reply.usage.completion_tokens
        msg = reply.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls:
            return msg.content, calls
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            result = run_tool(call.function.name, args)
            calls.append({"args": args, "result": result})
            if show:
                print(f"🔧 {call.function.name}({args}) → {str(result)[:90]}")
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": json.dumps(result, ensure_ascii=False)})
    return "stopped: step limit reached", calls

## 3. A vague tool

Name `search` · arguments `q` and `n` · description just "search"
(the tool's author knows `n` is the starting year — but the model only sees the schema)

In [ ]:
VAGUE = [{"type": "function", "function": {
    "name": "search",
    "description": "search",
    "parameters": {"type": "object",
                   "properties": {"q": {"type": "string"}, "n": {"type": "integer"}},
                   "required": ["q", "n"]}}}]

def run_vague(name, args):
    return find_papers(args["q"], year_from=args["n"])

In [ ]:
QUESTION = "Find papers on transformers for time series published in 2023 or later, and list all their titles."

def test(tools, run_tool, year_key, rounds=5):
    for i in range(rounds):
        answer, calls = run_agent(QUESTION, tools, run_tool, show=False)
        for c in calls:
            old = sum(p["year"] < 2023 for p in c["result"])
            print(f"round {i+1}: {year_key}={c['args'].get(year_key)} → got {len(c['result'])} papers, older than 2023: {old}")
test(VAGUE, run_vague, "n")

## 4. A clear tool

The name says what it searches · each argument says what it means · the description says what it returns, and what you get when nothing is found

In [ ]:
CLEAR = [{"type": "function", "function": {
    "name": "search_papers",
    "description": "Search academic papers in the Cogito Learn library by topic. Returns a list of {id, title, year}; returns [] if none match.",
    "parameters": {"type": "object", "properties": {
        "query": {"type": "string", "description": "Topic in English, e.g. 'time series transformer'"},
        # ✏️ TODO 1 (▶ video 4:50): Write year_from's description so the model knows which year it is and how it's used
        "year_from": {"type": "integer", "description": ___}},
    "required": ["query", "year_from"]}}}]

def run_clear(name, args):
    return find_papers(args["query"], year_from=args["year_from"])

test(CLEAR, run_clear, "year_from")

## 5. When the system fails: which error helps the model

`search_api` is a simulated search we can make fail: set `down["fails_left"]` = how many timeouts in a row

In [ ]:
class SearchTimeout(Exception):
    pass

down = {"fails_left": 0}

def search_api(query, year_from):
    if down["fails_left"] > 0:
        down["fails_left"] -= 1
        raise SearchTimeout("search API did not respond within 10 seconds")
    return find_papers(query, year_from)

In [ ]:
def run_naive(name, args):
    try:
        return search_api(args["query"], args["year_from"])
    except Exception:
        return "Error"

down["fails_left"] = 99
answer, calls = run_agent(QUESTION, CLEAR, run_naive)
print(answer)

In [ ]:
# ✏️ TODO 2 (▶ video 6:41): Catch only SearchTimeout and return a message telling the model what happened
def run_honest(name, args):
    try:
        return search_api(args["query"], args["year_from"])
    except ___:
        return ___

down["fails_left"] = 99
answer, calls = run_agent(QUESTION, CLEAR, run_honest)
print(answer)

## 6. Retry in the harness

Retry **only temporary errors** (timeout, 429) — wait before retrying, and wait longer each time (1 → 2 seconds)
Never retry a usage error (wrong arguments): the same input gives the same error (EP02 Lab 4)

In [ ]:
def with_retry(fn, *args, tries=3, wait=1):
    for attempt in range(1, tries + 1):
        try:
            return fn(*args)
        # ✏️ TODO 3 (▶ video 7:34): Catch only the temporary error; on the last attempt, raise it on
        except ___ as e:
            if ___:
                raise
            # ✏️ TODO 4 (▶ video 7:52): Wait before retrying, and double the wait each time
            print(f"   ⚠️ attempt {attempt}: {e} → waiting {wait} s, then retrying")
            ___
            ___

In [ ]:
def run_retry(name, args):
    try:
        return with_retry(search_api, args["query"], args["year_from"])
    except SearchTimeout:
        return "search_papers is temporarily unavailable (timeout) — this does not mean there are no papers"

down["fails_left"] = 2
answer, calls = run_agent(QUESTION, CLEAR, run_retry)
print(answer)

## 7. Graceful degradation: answer from a cache and say so

If retries run out and search is still down (and the backup route from EP05 Lab 4 is down too) — use an **old copy of the library** instead of not answering
This copy was saved on 15 Jan 2024, so it has no 2024 papers yet

In [ ]:
SNAPSHOT_DATE = "2024-01-15"
SNAPSHOT = [p for p in PAPERS if p["year"] <= 2023]
notes = []

# ✏️ TODO 5 (▶ video 9:05): If it's still down after the retries, save a warning in notes and search SNAPSHOT instead

def run_degrade(name, args):
    try:
        return with_retry(search_api, args["query"], args["year_from"])
    except SearchTimeout:
        notes.append(___)
        return {"source": f"library copy of {SNAPSHOT_DATE} — may be out of date",
                "papers": ___}

In [ ]:
down["fails_left"] = 99
answer, calls = run_agent(QUESTION, CLEAR, run_degrade)
print(answer)

In [ ]:
# ✏️ TODO 6 (▶ video 9:41): If notes has warnings, put them before the model's answer
def final_answer(answer):
    return ___

print(final_answer(answer))

In [ ]:
cost = USAGE["in"] / 1e6 * 0.07 + USAGE["out"] / 1e6 * 0.34
print(f"the whole lab used {USAGE['in']:,} input tokens + {USAGE['out']:,} output tokens ≈ ${cost:.4f}")

## Summary

| Principle | What we saw in the lab |
|---|---|
| ③ good tools | `search(q, n)` → the model passed `n=10` (guessing it was a result count) · `search_papers(query, year_from)` + descriptions → `year_from=2023` every round |
| ③ good errors | `"Error"` → the model rewrote its query, called again and suggested the user change their query · an error that states the cause → one call, and a plain answer to the user |
| ④ retry | retry only temporary errors, waiting 1 → 2 seconds · the model never knew it failed |
| ④ graceful degradation | answer from the library copy · warnings the user must see are **attached by the harness**, not left to the model |

**Exercises**
1. Delete the `description` of `year_from` in `CLEAR` (keep only the name) and run `test` again — is a good name alone enough?
2. Make `with_retry` take a list of exceptions to retry (e.g. timeout and 429) instead of hard-coding `SearchTimeout`

**Next — Lab 2:** an agent that fails, and knowing where it failed — add a **trace** to every step, then test with many kinds of input